# 03 — M5/M6: Action-conditioned world model ve CEM/MPC

Bu notebook representation'dan control'e geçişi görünür kılar:

`current latent + action/state history → future latent rollout → goal energy → CEM → ilk action → yeniden planla`

Hızlı ilk deney normalize 2D konumu kullanarak dynamics ve planner hatasını visual encoder hatasından ayıran bir **privileged-latent kontrol testi**dir. Ardından aynı scriptin `--latent-source visual` yolu planar RGB transition kliplerinde tiny video-JEPA'yı eğitir, encoder'ı dondurur ve AC predictor/CEM hattını goal-image latent'iyle uçtan uca çalıştırır.

In [ ]:
from pathlib import Path
import os, sys
override = os.environ.get('JEPA_LAB_ROOT')
candidates = ([Path(override)] if override else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/jepa-study'), Path('/kaggle/working/I-JEPA'), Path('/content/jepa-study'), Path('/content/I-JEPA')]
ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Repo bulunamadı; JEPA_LAB_ROOT değişkenini ayarlayın.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('repo:', ROOT)

In [ ]:
%matplotlib inline
from pprint import pprint
import subprocess
import numpy as np
import torch
import torch.nn.functional as F
from jepa_lab.device import seed_everything, select_device
from jepa_lab.runlog import load_yaml

seed_everything(42)
device = select_device('auto')
config = load_yaml(ROOT / 'configs/action_world_model.yaml')
print('device:', device)
pprint(config)

## V-JEPA2-AC resmî trajectory preflight

Bu adım model indirmez veya büyük modeli belleğe ayırmaz. Pinned `franka_example_traj.npz`, kaynak commit, trajectory tensor şekilleri, opsiyonel bağımlılıklar ve donanımı raporlar. VRAM `≥24 GB` değilse full replay zorunlu değildir.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/official_vjepa2_ac_replay.py'), '--preflight-only'], cwd=ROOT, check=True)

In [ ]:
trajectory_path = ROOT / 'upstream/vjepa2/notebooks/franka_example_traj.npz'
with np.load(trajectory_path, allow_pickle=False) as trajectory:
    trajectory_shapes = {name: {'shape': trajectory[name].shape, 'dtype': str(trajectory[name].dtype)} for name in trajectory.files}
pprint(trajectory_shapes)

## Tahmin 1 — Causal sınır ve action token'ları

Action formatı sabittir: `[dx,dy,dz,droll,dpitch,dyaw,dgripper]`. Planar simülatör yalnız `dx,dy` kullanır. Block-causal attention'da zaman bloğu `t=1`, `t=3` action/state bilgisini görebilir mi? Maskede `True` değerinin attention'ı engellediğini unutmayın.

In [ ]:
from jepa_lab.action_world_model import ActionWorldModel, BlockCausalPredictor

causal_mask = BlockCausalPredictor.block_causal_mask(4)
print(causal_mask.int())
assert causal_mask[1, 3] and not causal_mask[3, 1]

### İncele 1

Satır sorguyu, sütun anahtarı temsil eder. Üst üçgen maskeli olduğu için her blok yalnız kendisini ve geçmişi görür. Bu, non-causal V-JEPA masking'den control için gerekli temel ayrımdır.

## Teacher forcing ve autoregressive rollout

### Tahmin 2

Teacher forcing her adımda ground-truth önceki latent'i verir. Rollout ise kendi tahminini bir sonraki adıma taşır. Horizon uzadıkça iki loss neden ayrışır? Çıktı şekillerini tahmin edin: `B=8, H=4, N=1, D=2`.

In [ ]:
from jepa_lab.datasets import sample_planar_latent_batch

action_model = ActionWorldModel(
    latent_dim=2, action_dim=7, state_dim=7,
    hidden_dim=64, num_layers=2, num_heads=4, max_horizon=4,
).to(device)
z0, actions, states, targets = sample_planar_latent_batch(8, 4, seed=42, device=device)
initial_output = action_model.compute_loss(z0, actions, targets, states, rollout_steps=2)
print('teacher predictions:', tuple(initial_output.teacher_forcing_predictions.shape))
print('2-step rollout:', tuple(initial_output.rollout_predictions.shape))
print({'total': float(initial_output.loss.detach()), 'teacher': float(initial_output.teacher_forcing_loss.detach()), 'rollout': float(initial_output.rollout_loss.detach())})
assert initial_output.teacher_forcing_predictions.shape == (8, 4, 1, 2)
assert initial_output.rollout_predictions.shape == (8, 2, 1, 2)

## Hızlı action-conditioned dynamics eğitimi

Doğru action'ın latent L1 hatası zero ve shuffled action'dan düşük olmalıdır. Bu karşılaştırma modelin action'ı görmezden gelmediğine dair en doğrudan mekanizma testidir. CPU'da hızlı olması için width 64/2 layer; full config width 256/4 layer'dır.

In [ ]:
QUICK_ACTION_STEPS = 200  # İlk denemede 40, daha kararlı kıyas için 200-500
QUICK_BATCH = 128

In [ ]:
from jepa_lab.experiments import train_action_steps

action_opt = torch.optim.AdamW(action_model.parameters(), lr=3e-3)
action_metrics = train_action_steps(
    action_model, action_opt, steps=QUICK_ACTION_STEPS, batch_size=QUICK_BATCH,
    horizon=4, action_limit=0.10, device=device, seed=42,
)
pprint(action_metrics.as_dict())

### İncele 2

`correct < zero` ama `correct ≈ shuffled` ise model action büyüklüğünü kullanıp örneğe özgü eşleşmeyi öğrenmemiş olabilir. Üç koşulu birlikte raporlayın; yalnız train loss'u raporlamayın.

In [ ]:
z0_eval, a_eval, s_eval, y_eval = sample_planar_latent_batch(256, 4, seed=999, device=device)
rollout_errors = {}
with torch.inference_mode():
    for horizon in (1, 2, 4):
        pred = action_model.rollout(z0_eval, a_eval[:, :horizon], s_eval[:, :horizon])
        rollout_errors[horizon] = float(F.l1_loss(pred, y_eval[:, :horizon]))
print('1/2/4-step rollout L1:', rollout_errors)

## Önce oracle dynamics ile CEM'i izole et

### Tahmin 3

CEM 256 action sequence örnekler, en iyi 32 elite'i tutar ve dağılımı 5 kez daraltır. Elite ortalama enerjisi refinement boyunca artmalı mı, azalmalı mı? MPC neden yalnız ilk action'ı uygulayıp yeniden planlar?

In [ ]:
from jepa_lab.cem import CEMPlanner, receding_horizon_control
from jepa_lab.simulator import PlanarReachEnv
from jepa_lab.visualization import plot_planar_trace

oracle_env = PlanarReachEnv(image_size=64, max_delta=0.10, success_radius=0.08, max_steps=20, seed=7)
_, reset_info = oracle_env.reset(seed=7)
oracle_planner = CEMPlanner(
    horizon=4, action_dim=7, candidates=256, elites=32, refinements=5,
    action_low=oracle_env.action_low, action_high=oracle_env.action_high, seed=7,
)
one_plan = oracle_planner.plan(oracle_env.trajectory_cost)
print('elite energy history:', one_plan.energy_history)
assert all(a >= b for a, b in zip(one_plan.energy_history, one_plan.energy_history[1:]))

In [ ]:
oracle_env = PlanarReachEnv(image_size=64, max_delta=0.10, success_radius=0.08, max_steps=20, seed=7)
_, reset_info = oracle_env.reset(seed=7)
oracle_planner = CEMPlanner(
    horizon=4, action_dim=7, candidates=256, elites=32, refinements=5,
    action_low=oracle_env.action_low, action_high=oracle_env.action_high, seed=7,
)
trace = receding_horizon_control(oracle_env, oracle_planner, lambda env: env.trajectory_cost, max_steps=20)
positions = np.stack([reset_info['position'], *[step.info['position'] for step in trace]])
display(plot_planar_trace(positions, reset_info['goal']))
print({'success': trace[-1].info['success'], 'distance': trace[-1].info['distance'], 'steps': len(trace)})

## Learned latent dynamics ile reduced CEM

Bu kez candidate energy oracle simulator'dan değil, eğitilmiş action model rollout'undan gelir. Hızlı profil `64 candidate / 8 elite / 3 refinement`; kabul deneyi aşağıda `256/32/5` kullanır.

In [ ]:
from jepa_lab.experiments import run_learned_planar_episode

learned_episode = run_learned_planar_episode(
    action_model, seed=123, device=device, horizon=4,
    candidates=64, elites=8, refinements=3, max_steps=20,
)
pprint(learned_episode)
assert all(
    all(a >= b for a, b in zip(history, history[1:]))
    for history in learned_episode['energy_histories']
)

## Tam öğretici yol: RGB → tiny V-JEPA → frozen encoder → AC predictor → CEM

Analytic XY deneyi bir kontrol deneyidir. Aşağıdaki opt-in koşu ise planar RGB transition kliplerinde tiny video-JEPA'yı eğitir, EMA target encoder'ı dondurur, AC optimizer'dan dışlar, sonraki RGB latent'lerini öğretmen hedefi yapar ve her MPC adımında current/goal RGB'yi yeniden encode eder. Planner hedef koordinatını değil yalnız **goal-image latent**'ini cost olarak görür. Smoke sonucu acceptance iddiası değildir.

In [ ]:
RUN_VISUAL_SMOKE = False
if RUN_VISUAL_SMOKE:
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/planar_world_model_demo.py'),
        '--profile', 'smoke', '--latent-source', 'visual', '--device', str(device),
        '--output', str(ROOT / 'runs/planar_visual_smoke.json'),
    ], cwd=ROOT, check=True)
else:
    print('Atlandı. Önce analytic kontrolü açıklayın, sonra RUN_VISUAL_SMOKE=True yapın.')

## Full M6 kabul koşuları — Kaggle GPU

Aynı CLI iki deneyi açıkça ayırır: `analytic`, dynamics/CEM için privileged XY kontrolüdür; `visual`, RGB'den eğitilip dondurulan tiny video-JEPA latent'lerini kullanır. Full profil width 256, 4 layer, 8 head, 100 held-out seed ve `256/32/5` CEM kullanır. `--enforce` ancak ilgili koşunun `%80` başarı, random'a `30` puan marjı, monoton elite energy ve action-error kontrollerinin tümü geçerse sıfır döner. Visual yol deneysel olabilir; başarısız sonucu gizlemeyin.

In [ ]:
RUN_FULL_ANALYTIC_M6 = False
RUN_FULL_VISUAL_M6 = False

In [ ]:
for enabled, latent_source in ((RUN_FULL_ANALYTIC_M6, 'analytic'), (RUN_FULL_VISUAL_M6, 'visual')):
    if enabled:
        subprocess.run([
            sys.executable, str(ROOT / 'scripts/planar_world_model_demo.py'),
            '--profile', 'full', '--latent-source', latent_source, '--device', str(device),
            '--output', str(ROOT / f'runs/planar_{latent_source}_full.json'), '--enforce',
        ], cwd=ROOT, check=True)
if not RUN_FULL_ANALYTIC_M6 and not RUN_FULL_VISUAL_M6:
    print('Atlandı. Kaggle GPU hazır olduğunda seçtiğiniz bayrağı açın.')

## Resmî V-JEPA2-AC replay — manuel ve izole kernel

Pinned notebook: `upstream/vjepa2/notebooks/energy_landscape_example.ipynb`. Bu notebook current/goal latent, 125 noktalı xyz energy grid ve reduced CEM gösterir; ground-truth/zero/shuffled scalar energy kıyasını kendisi yapmaz. Aşağıdaki ayrı-process wrapper, doğrulanmış yerel checkpoint ile ground-truth ve zero enerjilerini ekler. Bundled dosyada yalnız bir transition bulunduğundan bilimsel olarak geçerli bir shuffled-action kontrolü yoktur ve çıktı bunu `unsupported` olarak kaydeder.

- VRAM `≥24 GB`: resmî GPU replay + reduced CEM.
- VRAM `<24 GB`: tam inference geçiş kriteri değildir. Yeterli sistem RAM'i varsa 25 candidate / 2 refinement CPU smoke denenebilir.

ViT-g + AC predictor yaklaşık 1.32B parametredir; 16 GB M4'te yalnız preflight çalıştırın. Candidate sayısı, horizon, action sınırları ve latency JSON çıktısında kalır; paper sonucu ile bu smoke testi aynı iddia değildir.

In [ ]:
RUN_OFFICIAL_AC_REPLAY = False  # Yalnız uygun GPU/RAM ve doğrulanmış checkpoint ile True
ac_checkpoint = ROOT / 'checkpoints/vjepa2-ac-vitg.pt'
ac_sha256 = os.environ.get('VJEPA2_AC_SHA256')  # sidecar varsa gerekli değildir
if RUN_OFFICIAL_AC_REPLAY:
    command = [
        sys.executable, str(ROOT / 'scripts/official_vjepa2_ac_replay.py'),
        '--run-replay', '--checkpoint', str(ac_checkpoint),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
        '--output', str(ROOT / 'runs/vjepa2_ac_official_replay.json'),
    ]
    if ac_sha256:
        command.extend(['--sha256', ac_sha256])
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Ağır replay atlandı; default preflight yukarıda tamamlandı.')

## M5/M6 geçiş kontrolü

- Frozen encoder representation üretir; causal AC predictor action/state ile gelecek latent'i tahmin eder.
- Teacher forcing ile autoregressive rollout arasındaki exposure/compounding-error farkını açıklayın.
- Doğru/zero/shuffled action enerjileri, modelin action'a gerçekten koşullandığını sınar.
- CEM açık çevrim sequence arar; MPC yalnız ilk action'ı uygular ve yeni gözlemle yeniden planlar.
- Privileged XY latent, RGB visual latent ve resmî V-JEPA2-AC sonuçlarını ayrı etiketleyin.
- Gerçek robot entegrasyonu için coordinate transform, workspace clipping, collision/safety ve controller hook'ları ayrıca gerekir.